# General N-Direction Random Walk Solution

This notebook simulates the random walk with N equally spaced directions:
- Choose k uniformly from {0, 1, ..., N-1}
- Use direction angle = 2*pi*k/N
- Take a unit step in that direction

The 4-direction case is exactly the mathemagicland model.

All files are saved in `outputs/general_walk_solution/`.

In [ ]:
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils import (
    ensure_output_dirs,
    plot_histogram,
    plot_overlay_2d,
    plot_single_walk_2d,
    save_2d_path_json,
    save_summary_csv,
    simulate_general_walk,
)

# Configuration
RANDOM_SEED = 20260407
N_TRIALS = 300
MAX_STEPS = 20_000
N_DIRECTIONS = 6
RETURN_TOLERANCE = 1e-9
POSITION_ROUND_DECIMALS = 12

# Simple assumption: run this notebook from inside random_walks/
NOTEBOOK_DIR = Path('.')
OUTPUT_DIR, PATHS_DIR = ensure_output_dirs(NOTEBOOK_DIR / 'outputs' / 'general_walk_solution')

plt.style.use('seaborn-v0_8-whitegrid')
rng = np.random.default_rng(RANDOM_SEED)

print(f'Working directory: {NOTEBOOK_DIR.resolve()}')
print(f'Output directory:  {OUTPUT_DIR.resolve()}')
print(f'N_DIRECTIONS:      {N_DIRECTIONS}')

## Mathematical Description and Robust Return Detection

Position after n steps is a sum of unit vectors chosen from N equally spaced angles.

Floating-point note:
- For many N values, coordinates are not exactly representable in binary floating point.
- Exact checks like x == 0 and y == 0 can miss true returns due to tiny numeric error.

Robust logic used here:
- For N in {2, 4}, use exact axis-aligned steps and exact return checks.
- Otherwise, use tolerance-based return detection (`RETURN_TOLERANCE`) and small rounding.

This keeps return detection stable while matching the intended model.

In [ ]:
def make_trial_record(walk: Dict[str, object], trial_index: int) -> Dict[str, object]:
    return {
        'trial_index': trial_index,
        'n_directions': N_DIRECTIONS,
        'steps_until_stop': walk['steps_until_stop'],
        'return_time': walk['return_time'],
        'returned_to_origin': walk['returned_to_origin'],
        'censored': walk['censored'],
        'max_distance': walk['max_distance'],
        'max_manhattan': walk['max_manhattan'],
        'bbox_width': walk['bbox_width'],
        'bbox_height': walk['bbox_height'],
        'bbox_area': walk['bbox_area'],
        'final_x': walk['final_x'],
        'final_y': walk['final_y'],
    }

## Single Example Walk

Simulate one walk using the configured `N_DIRECTIONS`.

In [ ]:
single_walk = simulate_general_walk(
    rng=rng,
    n_directions=N_DIRECTIONS,
    max_steps=MAX_STEPS,
    return_tolerance=RETURN_TOLERANCE,
    round_decimals=POSITION_ROUND_DECIMALS,
)

single_walk_plot_path = OUTPUT_DIR / 'single_walk.png'
plot_single_walk_2d(
    xs=single_walk['xs'],
    ys=single_walk['ys'],
    out_path=single_walk_plot_path,
    title=f'Single {N_DIRECTIONS}-Direction Walk',
)

print('Single walk summary:')
print(f"  steps_until_stop:   {single_walk['steps_until_stop']}")
print(f"  returned_to_origin: {single_walk['returned_to_origin']}")
print(f"  max_distance:       {single_walk['max_distance']:.3f}")
print(f'  saved plot:         {single_walk_plot_path}')

## Monte Carlo Simulation

Run many independent trials, save summary metrics, and save each raw path.

In [ ]:
trial_records: List[Dict[str, object]] = []
all_paths: List[Tuple[List[float], List[float]]] = []

for trial_idx in range(N_TRIALS):
    walk = simulate_general_walk(
        rng=rng,
        n_directions=N_DIRECTIONS,
        max_steps=MAX_STEPS,
        return_tolerance=RETURN_TOLERANCE,
        round_decimals=POSITION_ROUND_DECIMALS,
    )
    all_paths.append((walk['xs'], walk['ys']))
    save_2d_path_json(walk['xs'], walk['ys'], trial_idx, PATHS_DIR)
    trial_records.append(make_trial_record(walk, trial_idx))

summary_df = pd.DataFrame(trial_records)
summary_csv_path = save_summary_csv(summary_df, OUTPUT_DIR / 'summary.csv')

completed = summary_df[~summary_df['censored']]

print(f'Trials run:                {N_TRIALS}')
print(f'Completed returns:         {len(completed)}')
print(f"Censored trials:           {int(summary_df['censored'].sum())}")
if len(completed):
    print(f"Mean return time:          {completed['return_time'].mean():.3f}")
print(f"Mean max distance:         {summary_df['max_distance'].mean():.3f}")
print(f'Saved summary CSV:         {summary_csv_path}')

summary_df.head()

## Visualization of Monte Carlo Results

Plot return-time and max-distance histograms.

In [ ]:
completed_return_times = summary_df.loc[~summary_df['censored'], 'return_time']

return_hist_path = OUTPUT_DIR / 'return_time_histogram.png'
max_dist_hist_path = OUTPUT_DIR / 'max_distance_histogram.png'

plot_histogram(
    values=completed_return_times,
    out_path=return_hist_path,
    title='Return Time Histogram (Completed Trials)',
    xlabel='Steps until first return',
    color='tab:blue',
)

plot_histogram(
    values=summary_df['max_distance'],
    out_path=max_dist_hist_path,
    title='Max Euclidean Distance Histogram',
    xlabel='Max Euclidean distance during walk',
    color='tab:orange',
)

print(f'Saved: {return_hist_path}')
print(f'Saved: {max_dist_hist_path}')

## Final Overlay Plot

Overlay all simulated paths on one set of axes (equal aspect ratio).

In [ ]:
overlay_path = OUTPUT_DIR / 'overlay_walks.png'
plot_overlay_2d(
    all_paths=all_paths,
    out_path=overlay_path,
    title=f'Overlay of All {N_DIRECTIONS}-Direction Walks',
)
print(f'Saved: {overlay_path}')